# HW 5 — Applied: Zero-Shot Restoration with DDNM
**Modern Computer Vision** — Technion, Spring 2026

**Due:** July 19, 2026

## The idea — no training, just linear algebra + a diffusion prior

You have a **pretrained** diffusion model that knows how to dream up faces, and a **degraded** image: some pixels masked out, or a tiny low-resolution thumbnail. Can you restore it **without training anything**? Yes — this is **DDNM** (Denoising Diffusion Null-space Model, Wang et al. 2023).

Write the degradation as a known linear operator $A$ (a mask; an average-pooling). The measurement is $y = A\,x_{\text{true}}$. Split any candidate clean image into two orthogonal parts:

- the **range space** of $A$ — what the measurement *can* see, which we must keep equal to $y$;
- the **null space** of $A$ — what the measurement *cannot* see, which the diffusion prior is free to fill in.

At every denoising step we predict the clean image $\hat x_0$ and replace it by

$$\hat x_0 \;\leftarrow\; A^{+}y \;+\; (I - A^{+}A)\,\hat x_0 \;=\; \hat x_0 + A^{+}\big(y - A\,\hat x_0\big),$$

where $A^{+}$ is the pseudo-inverse. The first term pins the visible part to the data; the second keeps the model's guess only in the invisible part. That single projection, dropped into an ordinary DDPM sampler, turns a face generator into a zero-shot inpainter, super-resolver, **and** colorizer. The operators are chosen so $A^{+}$ is trivial: a binary mask is its own pseudo-inverse, and average-pooling's is nearest-neighbour upsampling.

## Setup ⚠️ needs a GPU

We use an **ungated, pixel-space** unconditional DDPM (`google/ddpm-celebahq-256`) — no login, no text encoder. Pixel space matters: the projection above is linear algebra on *pixels*.

In [ ]:
%pip -q install diffusers accelerate matplotlib

### Provided machinery (you don't need to edit this)

The pretrained model and scheduler, the schedule tensors, one plain DDPM step `posterior`, an unconditional sampler `ddpm_sample` (to make a clean face to degrade), and a `show` helper.

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
import diffusers; diffusers.utils.logging.set_verbosity_error()
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = UNet2DModel.from_pretrained('google/ddpm-celebahq-256', use_safetensors=False).to(device).eval()
sch = DDPMScheduler.from_pretrained('google/ddpm-celebahq-256')
T = sch.config.num_train_timesteps
abar = sch.alphas_cumprod.to(device); alphas = sch.alphas.to(device); betas = sch.betas.to(device)
H = model.config.sample_size
print('loaded', H, 'px DDPM,', T, 'steps on', device)

@torch.no_grad()
def posterior(x, x0, i, gen):
    """One DDPM ancestral step x_i -> x_{i-1} from the predicted clean image x0."""
    ab, ab_prev, beta, alpha = abar[i], abar[i - 1], betas[i], alphas[i]
    mean = (ab_prev.sqrt() * beta) / (1 - ab) * x0 + (alpha.sqrt() * (1 - ab_prev)) / (1 - ab) * x
    return mean + (beta * (1 - ab_prev) / (1 - ab)).sqrt() * torch.randn(x.shape, device=device, generator=gen)

@torch.no_grad()
def ddpm_sample(seed=0):
    """Plain unconditional sample -> a clean face we will then degrade and restore."""
    gen = torch.Generator(device).manual_seed(seed)
    x = torch.randn(1, 3, H, H, device=device, generator=gen)
    for i in reversed(range(T)):
        eps = model(x, torch.tensor([i], device=device)).sample
        x0 = ((x - (1 - abar[i]).sqrt() * eps) / abar[i].sqrt()).clamp(-1, 1)
        x = posterior(x, x0, i, gen) if i > 0 else x0
    return x0

def show(panels):
    fig, ax = plt.subplots(1, len(panels), figsize=(3.3 * len(panels), 3.5), squeeze=False)
    for a, (im, ttl) in zip(ax[0], panels):
        a.imshow(((im[0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).cpu().numpy()); a.axis('off'); a.set_title(ttl)
    plt.tight_layout(); plt.show()

Make one clean face to play with (this is the only full generation; ~1 min):

In [ ]:
gt = ddpm_sample(seed=0)
show([(gt, 'ground truth')])

## Your part — the DDNM core

Implement the reverse loop. It is an ordinary DDPM sampler (you built one from scratch already) plus **one** extra line: after predicting $\hat x_0$, apply the range/null-space projection $\hat x_0 \leftarrow \hat x_0 + A^{+}(y - A\hat x_0)$. Everything else (`model`, `posterior`, the schedule tensors) is provided above.

*Each restoration below runs the full reverse diffusion: about a minute on a GPU, with no progress bar — it has not hung, just let it finish.*

In [ ]:
@torch.no_grad()
def ddnm(A, Apinv, y, seed=0):
    """Zero-shot restoration by range/null-space projection (DDNM).

    A      : the known degradation operator (e.g. a mask, or average-pooling)
    Apinv  : its pseudo-inverse  A^+
    y       = A(x_true) : the measurement we must stay consistent with

    Reverse the diffusion from pure noise. At each step predict the clean image
    x0, then CORRECT it so it matches the measurement on the range of A while the
    diffusion prior keeps filling the null space:
        x0  <-  x0 + Apinv( y - A(x0) )          # == A^+ y + (I - A^+A) x0
    Finally take one ordinary DDPM step. Returns the restored image (1,3,H,H).
    """
    # TODO -- implement the DDNM reverse loop described above.
    #         The key line is the range/null-space projection of x0.
    raise NotImplementedError("ddnm")

## Task 1 — Inpainting

The degradation is a binary **mask** $A$: it keeps the visible pixels and zeros the hole. A 0/1 mask is its own pseudo-inverse ($A = A^{\top} = A^{+}$), so both `A` and `Apinv` are just multiplication by the mask. The measurement $y$ is the masked face; DDNM fills the hole.

In [ ]:
mask = torch.ones(1, 1, H, H, device=device)
mask[:, :, H // 3:2 * H // 3, H // 4:3 * H // 4] = 0     # knock out a central box
A = lambda z: mask * z
Apinv = lambda r: mask * r
y = A(gt)
rec = ddnm(A, Apinv, y, seed=1)
show([(gt, 'ground truth'), (y, 'masked input'), (rec, 'DDNM inpainting')])

## Task 2 — Super-resolution

Same `ddnm`, a different operator: $A$ is $s\times s$ **average-pooling** (the image is only seen as a tiny thumbnail). Its pseudo-inverse is **nearest-neighbour upsampling** ($A^{+} = s^2 A^{\top}$). Note we use average-pooling, *not* bicubic, precisely because its pseudo-inverse is this simple.

In [ ]:
s = 8
A = lambda z: F.avg_pool2d(z, s)
Apinv = lambda r: F.interpolate(r, scale_factor=s, mode='nearest')
y = A(gt)
rec = ddnm(A, Apinv, y, seed=1)
show([(gt, 'ground truth'), (Apinv(y), 'low-res input (%dx)' % s), (rec, 'DDNM super-res')])

## Task 3 — Colorization

One more operator, same `ddnm`: $A$ collapses the three colour channels to a single **grayscale** value (their mean). Its pseudo-inverse copies that gray value back into all three channels ($A^{+} = 3A^{\top}$). Now the *brightness* lives in the range space (fixed to the measurement) and all the *colour* lives in the null space, so the diffusion prior invents a plausible colouring of a black-and-white photo.

In [ ]:
A = lambda z: z.mean(1, keepdim=True)              # RGB -> grayscale (1 channel)
Apinv = lambda r: r.repeat(1, 3, 1, 1)             # grayscale -> RGB (copy to 3 channels)
y = A(gt)
rec = ddnm(A, Apinv, y, seed=1)
show([(gt, 'ground truth'), (Apinv(y), 'grayscale input'), (rec, 'DDNM colorization')])

---
That is the whole method: one linear-algebra projection turns an unconditional face model into a zero-shot inpainter, super-resolver, and colorizer, no training. Any degradation with an easy pseudo-inverse drops straight in — try a different mask shape, another pooling factor, or your own operator.

*Modern Computer Vision — Technion — Spring 2026*